In [1]:
import numpy as np
import pandas as pd
 
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
 
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
 
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix
)
 
import warnings
warnings.filterwarnings("ignore")
 
RANDOM_STATE = 42
 

In [2]:
df = pd.read_csv("cleaned_osteoporosis_dataset.csv")
 
# Id is just a row identifier, not a feature
# df = df.drop(columns=["Id"])
 
print("Shape:", df.shape)
print("\nTarget distribution:\n", df["Osteoporosis"].value_counts())
 

Shape: (1954, 11)

Target distribution:
 Osteoporosis
1    979
0    975
Name: count, dtype: int64


In [10]:
# none_means_no_cols = ["Alcohol Consumption", "Medical Conditions", "Medications"]
# for col in none_means_no_cols:
#     df[col] = df[col].fillna("None")
 
# print("\nRemaining nulls after fill:\n", df.isnull().sum().sum())
 

In [3]:
X = df.drop(columns=["Osteoporosis"])
y = df["Osteoporosis"]
 
numeric_features = ["Age"]
categorical_features = [c for c in X.columns if c not in numeric_features]
 
print("\nNumeric features:", numeric_features)
print("Categorical features:", categorical_features)


Numeric features: ['Age']
Categorical features: ['Smoking', 'Physical_Activity', 'Medications', 'Vitamin_D_Intake', 'Hormonal_Changes', 'Gender', 'Body_Weight', 'Prior_Fractures', 'Alcohol_Consumption']


In [4]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_temp
)
 
print("\nTraining Set   :", X_train.shape)
print("Validation Set :", X_val.shape)
print("Test Set       :", X_test.shape)
 


Training Set   : (1367, 10)
Validation Set : (293, 10)
Test Set       : (294, 10)


In [6]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
 
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
 
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])
 

In [7]:
models = {
    "Logistic Regression": LogisticRegression(random_state=RANDOM_STATE, max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(random_state=RANDOM_STATE),
    "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
    "SVM": SVC(random_state=RANDOM_STATE, probability=True),
    "KNN": KNeighborsClassifier(),
    "XGBoost": XGBClassifier(random_state=RANDOM_STATE, eval_metric="logloss")
}
 
results = []
for name, model in models.items():
    pipe = Pipeline(steps=[("preprocess", preprocessor), ("model", model)])
    pipe.fit(X_train, y_train)
    val_pred = pipe.predict(X_val)
 
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_val, val_pred),
        "Precision": precision_score(y_val, val_pred),
        "Recall": recall_score(y_val, val_pred),
        "F1 Score": f1_score(y_val, val_pred)
    })
 
results_df = pd.DataFrame(results).sort_values(by="Accuracy", ascending=False)
print("\nModel comparison on validation set:\n", results_df.to_string(index=False))
 


Model comparison on validation set:
               Model  Accuracy  Precision   Recall  F1 Score
  Gradient Boosting  0.877133   0.991150 0.761905  0.861538
            XGBoost  0.849829   0.918699 0.768707  0.837037
      Decision Tree  0.825939   0.852941 0.789116  0.819788
                SVM  0.825939   0.936364 0.700680  0.801556
Logistic Regression  0.819113   0.885246 0.734694  0.802974
      Random Forest  0.802048   0.844961 0.741497  0.789855
                KNN  0.791809   0.870690 0.687075  0.768061


In [8]:
rf_pipe = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", RandomForestClassifier(random_state=RANDOM_STATE))
])
 
param_grid = {
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [5, 10, 15, None],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": ["sqrt", "log2"]
}
 
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
 
grid = GridSearchCV(
    estimator=rf_pipe,
    param_grid=param_grid,
    cv=cv,
    scoring="f1",
    n_jobs=-1
)
grid.fit(X_train, y_train)
 
print("\nBest RF params:", grid.best_params_)
best_rf = grid.best_estimator_
 


Best RF params: {'model__max_depth': 15, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 2, 'model__min_samples_split': 5, 'model__n_estimators': 100}


In [9]:
xgb_pipe = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", XGBClassifier(random_state=RANDOM_STATE, eval_metric="logloss"))
])
 
xgb_param_grid = {
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [3, 4, 5, 6],
    "model__learning_rate": [0.01, 0.05, 0.1],
    "model__subsample": [0.8, 1.0],
    "model__colsample_bytree": [0.8, 1.0]
}
 
xgb_grid = GridSearchCV(
    estimator=xgb_pipe,
    param_grid=xgb_param_grid,
    cv=cv,
    scoring="f1",
    n_jobs=-1
)
xgb_grid.fit(X_train, y_train)
 
print("\nBest XGB params:", xgb_grid.best_params_)
best_xgb = xgb_grid.best_estimator_
 


Best XGB params: {'model__colsample_bytree': 0.8, 'model__learning_rate': 0.01, 'model__max_depth': 6, 'model__n_estimators': 100, 'model__subsample': 0.8}


In [10]:
gb_pipe = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", GradientBoostingClassifier(random_state=RANDOM_STATE))
])
 
gb_param_grid = {
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [2, 3, 4, 5],
    "model__learning_rate": [0.01, 0.05, 0.1],
    "model__subsample": [0.8, 1.0],
    "model__min_samples_leaf": [1, 2, 4]
}
 
gb_grid = GridSearchCV(
    estimator=gb_pipe,
    param_grid=gb_param_grid,
    cv=cv,
    scoring="f1",
    n_jobs=-1
)
gb_grid.fit(X_train, y_train)
 
print("\nBest Gradient Boosting params:", gb_grid.best_params_)
best_gb = gb_grid.best_estimator_
 


Best Gradient Boosting params: {'model__learning_rate': 0.05, 'model__max_depth': 2, 'model__min_samples_leaf': 1, 'model__n_estimators': 300, 'model__subsample': 0.8}


In [11]:
candidates = {
    "Random Forest (tuned)": best_rf,
    "XGBoost (tuned)": best_xgb,
    "Gradient Boosting (tuned)": best_gb
}
 
final_scores = []
for name, pipe in candidates.items():
    val_pred = pipe.predict(X_val)
    final_scores.append((name, accuracy_score(y_val, val_pred)))
    print(f"\n{name} — Validation Accuracy: {accuracy_score(y_val, val_pred):.4f}")
    print(classification_report(y_val, val_pred))
 
best_model_name = max(final_scores, key=lambda x: x[1])[0]
best_model = candidates[best_model_name]
print(f"\nSelected final model: {best_model_name}")
 


Random Forest (tuned) — Validation Accuracy: 0.8225
              precision    recall  f1-score   support

           0       0.76      0.95      0.84       146
           1       0.93      0.70      0.80       147

    accuracy                           0.82       293
   macro avg       0.84      0.82      0.82       293
weighted avg       0.84      0.82      0.82       293


XGBoost (tuned) — Validation Accuracy: 0.8771
              precision    recall  f1-score   support

           0       0.81      0.99      0.89       146
           1       0.99      0.76      0.86       147

    accuracy                           0.88       293
   macro avg       0.90      0.88      0.88       293
weighted avg       0.90      0.88      0.88       293


Gradient Boosting (tuned) — Validation Accuracy: 0.8805
              precision    recall  f1-score   support

           0       0.81      1.00      0.89       146
           1       1.00      0.76      0.86       147

    accuracy             

In [16]:
# Combine Train and Validation data

X_train_final = pd.concat([X_train, X_val], axis=0)
y_train_final = pd.concat([y_train, y_val], axis=0)

# Train best model again
best_model.fit(X_train_final, y_train_final)

print("Final model trained on Train + Validation data")

Final model trained on Train + Validation data


In [17]:
test_pred = best_model.predict(X_test)
test_proba = best_model.predict_proba(X_test)[:, 1]
 
print("\n--- FINAL TEST SET PERFORMANCE ---")
print("Accuracy :", accuracy_score(y_test, test_pred))
print("ROC-AUC  :", roc_auc_score(y_test, test_proba))
print(classification_report(y_test, test_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, test_pred))
 


--- FINAL TEST SET PERFORMANCE ---
Accuracy : 0.8809523809523809
ROC-AUC  : 0.883659586283493
              precision    recall  f1-score   support

           0       0.81      0.99      0.89       147
           1       0.99      0.77      0.87       147

    accuracy                           0.88       294
   macro avg       0.90      0.88      0.88       294
weighted avg       0.90      0.88      0.88       294

Confusion Matrix:
 [[146   1]
 [ 34 113]]


In [18]:
cv_scores = cross_val_score(best_model, X, y, cv=cv, scoring="accuracy")
print("\n5-Fold CV Accuracy (full data): {:.4f} +/- {:.4f}".format(cv_scores.mean(), cv_scores.std()))


5-Fold CV Accuracy (full data): 0.9135 +/- 0.0104


In [19]:
if hasattr(best_model.named_steps["model"], "feature_importances_"):
    feature_names = best_model.named_steps["preprocess"].get_feature_names_out()
    importances = pd.Series(
        best_model.named_steps["model"].feature_importances_,
        index=feature_names
    ).sort_values(ascending=False)
    print("\nTop 15 most important features:\n", importances.head(15))
 


Top 15 most important features:
 num__Age                      0.985690
cat__Body_Weight_0            0.001671
cat__Prior_Fractures_1        0.001501
cat__Body_Weight_1            0.001328
cat__Prior_Fractures_0        0.001254
cat__Hormonal_Changes_0       0.001076
cat__Medications_1            0.000975
cat__Smoking_1                0.000942
cat__Medications_0            0.000773
cat__Smoking_0                0.000683
cat__Vitamin_D_Intake_1       0.000623
cat__Gender_0                 0.000567
cat__Gender_1                 0.000506
cat__Alcohol_Consumption_1    0.000485
cat__Physical_Activity_0      0.000478
dtype: float64


In [21]:
import joblib

joblib.dump(best_model, "osteoporosis_model.pkl")

print("Model saved successfully!")

Model saved successfully!


In [22]:
import joblib

loaded_model = joblib.load("osteoporosis_model.pkl")

print("Model loaded successfully!")

Model loaded successfully!


In [26]:
sample = {
    "Age": 60,
    "Gender": "Female",
    "Hormonal_Changes": "Postmenopausal",
    "Family_History": "Yes",
    "Race/Ethnicity": "Caucasian",
    "Body_Weight": "Underweight",
    "Calcium_Intake": "Low",
    "Vitamin_D_Intake": "Low",
    "Physical_Activity": "Sedentary",
    "Smoking": "Yes",
    "Alcohol_Consumption": "Moderate",
    "Medical_Conditions": "None",
    "Medications": "None",
    "Prior_Fractures": "No"
}

prediction = loaded_model.predict(pd.DataFrame([sample]))
probability = loaded_model.predict_proba(pd.DataFrame([sample]))[0][1]

print("Prediction:", prediction[0])
print("Risk:", probability)

Prediction: 1
Risk: 0.9972127165441409
